## 1. Core Data Structures

First, we'll define the data structures for representing ReAct traces.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Callable
from abc import ABC, abstractmethod
from enum import Enum
import json
import re

class StepType(Enum):
    """Types of steps in a ReAct trace."""
    THOUGHT = "thought"
    ACTION = "action"
    OBSERVATION = "observation"
    ANSWER = "answer"

@dataclass
class ReActStep:
    """A single step in the ReAct reasoning trace."""
    step_type: StepType
    content: str
    step_number: int
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def __str__(self) -> str:
        """Format the step for display."""
        type_name = self.step_type.value.capitalize()
        return f"{type_name} {self.step_number}: {self.content}"

@dataclass
class ActionCall:
    """Represents a parsed action call."""
    tool_name: str
    parameters: Dict[str, Any]
    raw_text: str
    
    def __str__(self) -> str:
        params_str = ", ".join(f"{k}={v}" for k, v in self.parameters.items())
        return f"{self.tool_name}({params_str})"

@dataclass
class ReActTrace:
    """Complete trace of a ReAct agent's reasoning and actions."""
    question: str
    steps: List[ReActStep] = field(default_factory=list)
    final_answer: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def add_thought(self, content: str, metadata: Optional[Dict[str, Any]] = None) -> None:
        """Add a thought step to the trace."""
        step_num = len([s for s in self.steps if s.step_type == StepType.THOUGHT]) + 1
        self.steps.append(ReActStep(
            step_type=StepType.THOUGHT,
            content=content,
            step_number=step_num,
            metadata=metadata or {}
        ))
    
    def add_action(self, content: str, metadata: Optional[Dict[str, Any]] = None) -> None:
        """Add an action step to the trace."""
        step_num = len([s for s in self.steps if s.step_type == StepType.ACTION]) + 1
        self.steps.append(ReActStep(
            step_type=StepType.ACTION,
            content=content,
            step_number=step_num,
            metadata=metadata or {}
        ))
    
    def add_observation(self, content: str, metadata: Optional[Dict[str, Any]] = None) -> None:
        """Add an observation step to the trace."""
        step_num = len([s for s in self.steps if s.step_type == StepType.OBSERVATION]) + 1
        self.steps.append(ReActStep(
            step_type=StepType.OBSERVATION,
            content=content,
            step_number=step_num,
            metadata=metadata or {}
        ))
    
    def set_final_answer(self, answer: str) -> None:
        """Set the final answer."""
        self.final_answer = answer
        self.steps.append(ReActStep(
            step_type=StepType.ANSWER,
            content=answer,
            step_number=1,
            metadata={}
        ))
    
    def get_thoughts(self) -> List[ReActStep]:
        """Get all thought steps."""
        return [s for s in self.steps if s.step_type == StepType.THOUGHT]
    
    def get_actions(self) -> List[ReActStep]:
        """Get all action steps."""
        return [s for s in self.steps if s.step_type == StepType.ACTION]
    
    def get_observations(self) -> List[ReActStep]:
        """Get all observation steps."""
        return [s for s in self.steps if s.step_type == StepType.OBSERVATION]
    
    def __str__(self) -> str:
        """Format the complete trace for display."""
        lines = [f"Question: {self.question}", ""]
        for step in self.steps:
            lines.append(str(step))
        return "\n".join(lines)

print("✓ ReAct data structures defined")

## 2. Action Parser

Parse actions from LLM output in various formats.

In [ ]:
class ActionParser:
    """Parses action calls from text in multiple formats."""
    
    @staticmethod
    def parse_bracket_format(text: str) -> Optional[ActionCall]:
        """Parse action in format: Action[tool_name]{param1: value1, param2: value2}"""
        pattern = r'Action\[([^\]]+)\]\{([^}]+)\}'
        match = re.search(pattern, text)
        if not match:
            return None
        
        tool_name = match.group(1).strip()
        params_str = match.group(2)
        
        # Parse parameters
        parameters = {}
        for param in params_str.split(','):
            if ':' in param:
                key, value = param.split(':', 1)
                parameters[key.strip()] = value.strip()
        
        return ActionCall(
            tool_name=tool_name,
            parameters=parameters,
            raw_text=text
        )
    
    @staticmethod
    def parse_function_format(text: str) -> Optional[ActionCall]:
        """Parse action in format: tool_name(param1=value1, param2=value2)"""
        pattern = r'(\w+)\(([^)]*)\)'
        match = re.search(pattern, text)
        if not match:
            return None
        
        tool_name = match.group(1)
        params_str = match.group(2)
        
        # Parse parameters
        parameters = {}
        if params_str.strip():
            for param in params_str.split(','):
                if '=' in param:
                    key, value = param.split('=', 1)
                    parameters[key.strip()] = value.strip().strip('"\'')
        
        return ActionCall(
            tool_name=tool_name,
            parameters=parameters,
            raw_text=text
        )
    
    @staticmethod
    def parse_json_format(text: str) -> Optional[ActionCall]:
        """Parse action in JSON format: {"tool": "name", "parameters": {...}}"""
        try:
            # Extract JSON from text
            json_match = re.search(r'\{[^}]+\}', text)
            if not json_match:
                return None
            
            data = json.loads(json_match.group())
            return ActionCall(
                tool_name=data.get('tool', ''),
                parameters=data.get('parameters', {}),
                raw_text=text
            )
        except json.JSONDecodeError:
            return None
    
    @classmethod
    def parse(cls, text: str) -> Optional[ActionCall]:
        """Try all parsing methods in order."""
        for parser in [cls.parse_bracket_format, cls.parse_function_format, cls.parse_json_format]:
            result = parser(text)
            if result:
                return result
        return None

# Test the parser
test_actions = [
    "Action[search]{query: Python programming}",
    "search(query='artificial intelligence')",
    '{"tool": "calculator", "parameters": {"expression": "2+2"}}'
]

print("Testing ActionParser:")
for action_text in test_actions:
    parsed = ActionParser.parse(action_text)
    print(f"  Input: {action_text}")
    print(f"  Parsed: {parsed}\n")

## 3. Tool System Integration

Define a simple tool system for the ReAct agent to use.

In [ ]:
class Tool(ABC):
    """Base class for all tools."""
    
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    @abstractmethod
    def execute(self, **kwargs) -> str:
        """Execute the tool with given parameters."""
        pass
    
    def get_description(self) -> str:
        """Get tool description for the prompt."""
        return f"{self.name}: {self.description}"

class SearchTool(Tool):
    """Mock search tool that returns predefined results."""
    
    def __init__(self):
        super().__init__(
            name="search",
            description="Search for information on the internet. Use query parameter."
        )
        self.knowledge_base = {
            "python": "Python is a high-level programming language known for its simplicity and readability.",
            "artificial intelligence": "AI is the simulation of human intelligence by machines, especially computer systems.",
            "react pattern": "ReAct is a paradigm that combines reasoning and acting in language models for better task solving.",
            "machine learning": "Machine learning is a subset of AI that enables systems to learn from data without explicit programming.",
            "eiffel tower height": "The Eiffel Tower is 330 meters (1,083 feet) tall, including antennas.",
            "population paris": "Paris has a population of approximately 2.2 million people within the city limits.",
        }
    
    def execute(self, query: str = "", **kwargs) -> str:
        """Execute search."""
        query_lower = query.lower()
        
        # Find best match
        for key, value in self.knowledge_base.items():
            if key in query_lower or any(word in query_lower for word in key.split()):
                return value
        
        return f"No information found for '{query}'."

class CalculatorTool(Tool):
    """Calculator tool for mathematical operations."""
    
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Perform mathematical calculations. Use expression parameter (e.g., '2+2', '10*5')."
        )
    
    def execute(self, expression: str = "", **kwargs) -> str:
        """Execute calculation."""
        try:
            # Safe evaluation - only allow basic math
            allowed_chars = set('0123456789+-*/().')
            if not all(c in allowed_chars or c.isspace() for c in expression):
                return "Error: Invalid characters in expression"
            
            result = eval(expression, {"__builtins__": {}}, {})
            return str(result)
        except Exception as e:
            return f"Error: {str(e)}"

class WikipediaTool(Tool):
    """Mock Wikipedia lookup tool."""
    
    def __init__(self):
        super().__init__(
            name="wikipedia",
            description="Look up information on Wikipedia. Use topic parameter."
        )
        self.articles = {
            "albert einstein": "Albert Einstein (1879-1955) was a theoretical physicist who developed the theory of relativity. He won the Nobel Prize in Physics in 1921.",
            "quantum mechanics": "Quantum mechanics is a fundamental theory in physics that describes nature at the scale of atoms and subatomic particles.",
            "world war ii": "World War II (1939-1945) was a global war involving most of the world's nations, forming two opposing alliances.",
            "photosynthesis": "Photosynthesis is the process by which plants use sunlight to synthesize nutrients from carbon dioxide and water.",
        }
    
    def execute(self, topic: str = "", **kwargs) -> str:
        """Execute Wikipedia lookup."""
        topic_lower = topic.lower()
        
        for key, value in self.articles.items():
            if key in topic_lower or topic_lower in key:
                return value
        
        return f"No Wikipedia article found for '{topic}'."

class ToolRegistry:
    """Registry for managing available tools."""
    
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
    
    def register(self, tool: Tool) -> None:
        """Register a tool."""
        self.tools[tool.name] = tool
    
    def get_tool(self, name: str) -> Optional[Tool]:
        """Get a tool by name."""
        return self.tools.get(name)
    
    def execute(self, name: str, **kwargs) -> str:
        """Execute a tool by name."""
        tool = self.get_tool(name)
        if not tool:
            return f"Error: Tool '{name}' not found"
        return tool.execute(**kwargs)
    
    def get_tool_descriptions(self) -> str:
        """Get descriptions of all registered tools."""
        descriptions = [tool.get_description() for tool in self.tools.values()]
        return "\n".join(descriptions)

# Create and populate tool registry
registry = ToolRegistry()
registry.register(SearchTool())
registry.register(CalculatorTool())
registry.register(WikipediaTool())

print("Available tools:")
print(registry.get_tool_descriptions())

## 4. Prompt Templates

Create prompt templates for the ReAct agent.

In [ ]:
class ReActPromptTemplate:
    """Template for ReAct prompts."""
    
    SYSTEM_TEMPLATE = """You are a helpful AI assistant that uses the ReAct (Reasoning + Acting) pattern to solve problems.

You have access to the following tools:
{tools}

Use the following format:

Question: the input question you must answer
Thought 1: you should always think about what to do
Action 1: the action to take, should be one of [{tool_names}]
Observation 1: the result of the action
... (this Thought/Action/Observation can repeat N times)
Thought N: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
"""
    
    USER_TEMPLATE = """Question: {question}
"""
    
    CONTINUE_TEMPLATE = """Observation {step}: {observation}
"""
    
    @classmethod
    def format_system_prompt(cls, tools: str, tool_names: str) -> str:
        """Format the system prompt with tool information."""
        return cls.SYSTEM_TEMPLATE.format(
            tools=tools,
            tool_names=tool_names
        )
    
    @classmethod
    def format_user_prompt(cls, question: str) -> str:
        """Format the initial user prompt."""
        return cls.USER_TEMPLATE.format(question=question)
    
    @classmethod
    def format_continue_prompt(cls, step: int, observation: str) -> str:
        """Format the continuation prompt after an observation."""
        return cls.CONTINUE_TEMPLATE.format(step=step, observation=observation)

# Example prompt
tools_desc = registry.get_tool_descriptions()
tool_names = ", ".join(registry.tools.keys())

system_prompt = ReActPromptTemplate.format_system_prompt(tools_desc, tool_names)
print("System Prompt:")
print(system_prompt)

## 5. Mock LLM for Testing

Create a mock LLM that simulates ReAct-style responses.

In [ ]:
class MockLLM:
    """Mock LLM that simulates ReAct reasoning."""
    
    def __init__(self):
        self.call_count = 0
        self.response_patterns = {
            "height of eiffel tower": [
                "Thought 1: I need to search for information about the Eiffel Tower's height.\nAction 1: search(query='Eiffel Tower height')",
                "Thought 2: I found the information. The Eiffel Tower is 330 meters tall.\nFinal Answer: The Eiffel Tower is 330 meters (1,083 feet) tall, including antennas."
            ],
            "what is 15 * 24": [
                "Thought 1: I need to calculate 15 multiplied by 24.\nAction 1: calculator(expression='15*24')",
                "Thought 2: The calculator shows the result is 360.\nFinal Answer: 15 * 24 = 360"
            ],
            "when did einstein win nobel prize": [
                "Thought 1: I should look up Albert Einstein on Wikipedia to find when he won the Nobel Prize.\nAction 1: wikipedia(topic='Albert Einstein')",
                "Thought 2: According to Wikipedia, Einstein won the Nobel Prize in Physics in 1921.\nFinal Answer: Albert Einstein won the Nobel Prize in Physics in 1921."
            ],
            "population of paris times 2": [
                "Thought 1: First, I need to find the population of Paris.\nAction 1: search(query='population Paris')",
                "Thought 2: Paris has about 2.2 million people. Now I need to multiply this by 2.\nAction 2: calculator(expression='2.2*2')",
                "Thought 3: The result is 4.4 million.\nFinal Answer: The population of Paris multiplied by 2 is approximately 4.4 million."
            ]
        }
    
    def generate(self, prompt: str, context: Optional[List[str]] = None) -> str:
        """Generate a response based on the prompt."""
        self.call_count += 1
        
        # Extract question from prompt
        question_match = re.search(r'Question: (.+)', prompt)
        if not question_match:
            return "Thought 1: I don't understand the question.\nFinal Answer: I cannot answer this question."
        
        question = question_match.group(1).strip().lower()
        
        # Find matching pattern
        for key, responses in self.response_patterns.items():
            if key in question:
                # Count how many observations we've already seen
                obs_count = prompt.count("Observation")
                if obs_count < len(responses):
                    return responses[obs_count]
        
        # Default response
        return "Thought 1: I don't have enough information to answer this question.\nFinal Answer: I cannot answer this question with the available tools."

# Test the mock LLM
llm = MockLLM()
test_prompt = ReActPromptTemplate.format_user_prompt("What is the height of the Eiffel Tower?")
response = llm.generate(test_prompt)
print("LLM Response:")
print(response)

## 6. ReAct Agent

Now let's build the complete ReAct agent that orchestrates the reasoning loop.

In [ ]:
class ReActAgent:
    """ReAct agent that combines reasoning and acting."""
    
    def __init__(self, llm, tool_registry: ToolRegistry, max_iterations: int = 10):
        self.llm = llm
        self.tool_registry = tool_registry
        self.max_iterations = max_iterations
        self.action_parser = ActionParser()
    
    def _parse_response(self, response: str) -> Dict[str, Any]:
        """Parse LLM response into thoughts, actions, and final answer."""
        result = {
            'thoughts': [],
            'actions': [],
            'final_answer': None
        }
        
        # Extract thoughts
        thought_pattern = r'Thought \d+: (.+?)(?=\n(?:Action|Thought|Final Answer)|$)'
        thoughts = re.findall(thought_pattern, response, re.DOTALL)
        result['thoughts'] = [t.strip() for t in thoughts]
        
        # Extract actions
        action_pattern = r'Action \d+: (.+?)(?=\n|$)'
        actions = re.findall(action_pattern, response)
        result['actions'] = [a.strip() for a in actions]
        
        # Extract final answer
        answer_pattern = r'Final Answer: (.+?)(?=\n|$)'
        answer_match = re.search(answer_pattern, response, re.DOTALL)
        if answer_match:
            result['final_answer'] = answer_match.group(1).strip()
        
        return result
    
    def _execute_action(self, action_text: str) -> str:
        """Execute an action and return the observation."""
        # Parse the action
        action_call = self.action_parser.parse(action_text)
        if not action_call:
            return f"Error: Could not parse action '{action_text}'"
        
        # Execute the tool
        try:
            result = self.tool_registry.execute(action_call.tool_name, **action_call.parameters)
            return result
        except Exception as e:
            return f"Error executing {action_call.tool_name}: {str(e)}"
    
    def run(self, question: str, verbose: bool = True) -> ReActTrace:
        """Run the ReAct agent on a question."""
        # Initialize trace
        trace = ReActTrace(question=question)
        
        # Build system prompt
        tools_desc = self.tool_registry.get_tool_descriptions()
        tool_names = ", ".join(self.tool_registry.tools.keys())
        system_prompt = ReActPromptTemplate.format_system_prompt(tools_desc, tool_names)
        
        # Start with the question
        prompt = system_prompt + ReActPromptTemplate.format_user_prompt(question)
        
        if verbose:
            print(f"Question: {question}\n")
        
        # ReAct loop
        for iteration in range(self.max_iterations):
            # Get LLM response
            response = self.llm.generate(prompt)
            
            # Parse response
            parsed = self._parse_response(response)
            
            # Add thoughts to trace
            for thought in parsed['thoughts']:
                trace.add_thought(thought)
                if verbose:
                    print(f"💭 Thought: {thought}")
            
            # Check for final answer
            if parsed['final_answer']:
                trace.set_final_answer(parsed['final_answer'])
                if verbose:
                    print(f"\n✅ Final Answer: {parsed['final_answer']}")
                break
            
            # Execute actions and get observations
            if parsed['actions']:
                for action_text in parsed['actions']:
                    trace.add_action(action_text)
                    if verbose:
                        print(f"🔧 Action: {action_text}")
                    
                    # Execute and observe
                    observation = self._execute_action(action_text)
                    trace.add_observation(observation)
                    if verbose:
                        print(f"👁️  Observation: {observation}\n")
                    
                    # Add observation to prompt for next iteration
                    prompt += f"\n{response}\n"
                    prompt += ReActPromptTemplate.format_continue_prompt(
                        len(trace.get_observations()),
                        observation
                    )
            else:
                # No actions, but no final answer either - something went wrong
                trace.set_final_answer("Unable to determine the answer.")
                if verbose:
                    print("⚠️  No action specified and no final answer. Stopping.")
                break
        
        if not trace.final_answer:
            trace.set_final_answer("Maximum iterations reached without finding an answer.")
            if verbose:
                print(f"⚠️  Reached maximum iterations ({self.max_iterations})")
        
        return trace

print("✓ ReActAgent defined")

## 7. Testing the ReAct Agent

Let's test our ReAct agent with various questions.

In [ ]:
# Create agent
agent = ReActAgent(llm=llm, tool_registry=registry, max_iterations=10)

# Test 1: Simple search
print("=" * 80)
print("TEST 1: Simple Search")
print("=" * 80)
trace1 = agent.run("What is the height of the Eiffel Tower?")
print("\n")

In [ ]:
# Test 2: Simple calculation
print("=" * 80)
print("TEST 2: Simple Calculation")
print("=" * 80)
trace2 = agent.run("What is 15 * 24?")
print("\n")

In [ ]:
# Test 3: Wikipedia lookup
print("=" * 80)
print("TEST 3: Wikipedia Lookup")
print("=" * 80)
trace3 = agent.run("When did Einstein win the Nobel Prize?")
print("\n")

In [ ]:
# Test 4: Multi-step reasoning
print("=" * 80)
print("TEST 4: Multi-step Reasoning")
print("=" * 80)
trace4 = agent.run("What is the population of Paris times 2?")
print("\n")

## 8. Trace Analysis

Analyze the traces to understand the agent's reasoning.

In [ ]:
def analyze_trace(trace: ReActTrace) -> Dict[str, Any]:
    """Analyze a ReAct trace."""
    return {
        'question': trace.question,
        'num_thoughts': len(trace.get_thoughts()),
        'num_actions': len(trace.get_actions()),
        'num_observations': len(trace.get_observations()),
        'total_steps': len(trace.steps),
        'final_answer': trace.final_answer,
        'success': trace.final_answer is not None and 'Error' not in trace.final_answer
    }

# Analyze all traces
traces = [trace1, trace2, trace3, trace4]
print("Trace Analysis:")
print("=" * 80)

for i, trace in enumerate(traces, 1):
    analysis = analyze_trace(trace)
    print(f"\nTrace {i}:")
    print(f"  Question: {analysis['question']}")
    print(f"  Thoughts: {analysis['num_thoughts']}")
    print(f"  Actions: {analysis['num_actions']}")
    print(f"  Observations: {analysis['num_observations']}")
    print(f"  Total Steps: {analysis['total_steps']}")
    print(f"  Success: {'✓' if analysis['success'] else '✗'}")

## 9. Advanced Features

Let's add some advanced features to enhance our ReAct agent.

In [ ]:
class EnhancedReActAgent(ReActAgent):
    """Enhanced ReAct agent with additional features."""
    
    def __init__(self, llm, tool_registry: ToolRegistry, max_iterations: int = 10):
        super().__init__(llm, tool_registry, max_iterations)
        self.trace_history: List[ReActTrace] = []
    
    def run(self, question: str, verbose: bool = True) -> ReActTrace:
        """Run the agent and store trace in history."""
        trace = super().run(question, verbose)
        self.trace_history.append(trace)
        return trace
    
    def get_statistics(self) -> Dict[str, Any]:
        """Get statistics about all runs."""
        if not self.trace_history:
            return {}
        
        total_thoughts = sum(len(t.get_thoughts()) for t in self.trace_history)
        total_actions = sum(len(t.get_actions()) for t in self.trace_history)
        total_observations = sum(len(t.get_observations()) for t in self.trace_history)
        successful = sum(1 for t in self.trace_history if t.final_answer and 'Error' not in t.final_answer)
        
        return {
            'total_runs': len(self.trace_history),
            'successful_runs': successful,
            'success_rate': successful / len(self.trace_history) if self.trace_history else 0,
            'avg_thoughts_per_run': total_thoughts / len(self.trace_history),
            'avg_actions_per_run': total_actions / len(self.trace_history),
            'avg_observations_per_run': total_observations / len(self.trace_history),
        }
    
    def explain_reasoning(self, trace: ReActTrace) -> str:
        """Generate a natural language explanation of the reasoning process."""
        lines = [f"To answer '{trace.question}', I followed these steps:\n"]
        
        step_num = 1
        thoughts = trace.get_thoughts()
        actions = trace.get_actions()
        observations = trace.get_observations()
        
        for i in range(max(len(thoughts), len(actions), len(observations))):
            if i < len(thoughts):
                lines.append(f"{step_num}. I thought: {thoughts[i].content}")
                step_num += 1
            if i < len(actions):
                lines.append(f"{step_num}. I took action: {actions[i].content}")
                step_num += 1
            if i < len(observations):
                lines.append(f"{step_num}. I observed: {observations[i].content}")
                step_num += 1
        
        if trace.final_answer:
            lines.append(f"\nFinal conclusion: {trace.final_answer}")
        
        return "\n".join(lines)
    
    def visualize_trace(self, trace: ReActTrace) -> str:
        """Create a simple ASCII visualization of the trace."""
        lines = []
        lines.append("┌─────────────────────────────────────────────┐")
        lines.append(f"│ Question: {trace.question[:40]:<40} │")
        lines.append("└─────────────────────────────────────────────┘")
        lines.append("")
        
        for step in trace.steps:
            if step.step_type == StepType.THOUGHT:
                symbol = "💭"
            elif step.step_type == StepType.ACTION:
                symbol = "🔧"
            elif step.step_type == StepType.OBSERVATION:
                symbol = "👁️"
            else:  # ANSWER
                symbol = "✅"
            
            content = step.content[:60]
            lines.append(f"{symbol} {step.step_type.value.capitalize()}: {content}")
        
        return "\n".join(lines)

# Test enhanced agent
enhanced_agent = EnhancedReActAgent(llm=llm, tool_registry=registry)

print("Testing Enhanced Agent:")
print("=" * 80)
test_trace = enhanced_agent.run("What is 15 * 24?", verbose=False)

print("\nExplanation:")
print(enhanced_agent.explain_reasoning(test_trace))

print("\nVisualization:")
print(enhanced_agent.visualize_trace(test_trace))

## 10. Error Handling and Recovery

Add robust error handling to the agent.

In [ ]:
class RobustReActAgent(EnhancedReActAgent):
    """ReAct agent with robust error handling."""
    
    def __init__(self, llm, tool_registry: ToolRegistry, max_iterations: int = 10, max_retries: int = 3):
        super().__init__(llm, tool_registry, max_iterations)
        self.max_retries = max_retries
    
    def _execute_action_with_retry(self, action_text: str) -> str:
        """Execute an action with retry logic."""
        last_error = None
        
        for attempt in range(self.max_retries):
            try:
                return self._execute_action(action_text)
            except Exception as e:
                last_error = str(e)
                if attempt < self.max_retries - 1:
                    continue
        
        return f"Error after {self.max_retries} attempts: {last_error}"
    
    def validate_trace(self, trace: ReActTrace) -> Dict[str, Any]:
        """Validate a trace for completeness and correctness."""
        issues = []
        
        # Check if we have a final answer
        if not trace.final_answer:
            issues.append("No final answer provided")
        
        # Check if actions have corresponding observations
        num_actions = len(trace.get_actions())
        num_observations = len(trace.get_observations())
        if num_actions != num_observations:
            issues.append(f"Action/Observation mismatch: {num_actions} actions, {num_observations} observations")
        
        # Check for error messages in observations
        error_observations = [obs for obs in trace.get_observations() if 'Error' in obs.content]
        if error_observations:
            issues.append(f"Found {len(error_observations)} error observations")
        
        return {
            'valid': len(issues) == 0,
            'issues': issues,
            'score': 1.0 - (len(issues) / 10.0)  # Simple scoring
        }

# Test robust agent
robust_agent = RobustReActAgent(llm=llm, tool_registry=registry)

print("Testing Robust Agent:")
print("=" * 80)
test_trace = robust_agent.run("What is the height of the Eiffel Tower?", verbose=False)

validation = robust_agent.validate_trace(test_trace)
print(f"\nTrace Validation:")
print(f"  Valid: {validation['valid']}")
print(f"  Score: {validation['score']:.2f}")
if validation['issues']:
    print(f"  Issues:")
    for issue in validation['issues']:
        print(f"    - {issue}")

## 11. Summary and Key Takeaways

### What We Built

1. **Core Data Structures**: ReActStep, ActionCall, ReActTrace for representing reasoning traces
2. **Action Parser**: Multi-format parser for extracting tool calls from LLM output
3. **Tool System**: Registry of tools (search, calculator, Wikipedia) that the agent can use
4. **Prompt Templates**: Structured prompts for guiding the LLM through ReAct reasoning
5. **ReAct Agent**: Core agent that orchestrates the Thought → Action → Observation loop
6. **Enhanced Features**: Trace history, statistics, explanations, and visualizations
7. **Error Handling**: Robust execution with retries and validation

### Key Concepts

- **ReAct Pattern**: Combines reasoning (thinking) with acting (tool use)
- **Interpretability**: Clear traces show how the agent thinks and acts
- **Grounding**: Actions and observations reduce hallucination
- **Modularity**: Separate components for parsing, execution, and reasoning

### Real-World Usage

To use this with a real LLM:

1. Replace `MockLLM` with an actual LLM client (OpenAI, Anthropic, etc.)
2. Replace mock tools with real implementations (actual search API, etc.)
3. Add more sophisticated parsing for handling various LLM output formats
4. Implement proper error handling and retry logic for API calls
5. Add logging and monitoring for production use

### Extensions

- **Multi-Agent ReAct**: Multiple agents working together
- **Reflection**: Agent reflects on its reasoning to improve
- **Planning**: Add explicit planning before action execution
- **Memory**: Integrate with memory systems for long-term context
- **Self-Correction**: Agent identifies and corrects its own errors

## 12. Practice Exercises

Try these exercises to deepen your understanding:

1. **Add a New Tool**: Create a weather tool or translation tool
2. **Improve Parsing**: Handle more complex action formats
3. **Add Reflection**: After each action, have the agent reflect on whether it helped
4. **Implement Planning**: Add a planning phase before the first action
5. **Multi-Step Validation**: Verify intermediate steps during execution
6. **Integrate Memory**: Use the memory system from notebook 06 to remember past queries
7. **Parallel Actions**: Execute multiple actions in parallel when possible
8. **Self-Healing**: If an action fails, try an alternative approach

### Exercise Template

In [ ]:
# Exercise: Add a new tool
class WeatherTool(Tool):
    """Mock weather lookup tool."""
    
    def __init__(self):
        super().__init__(
            name="weather",
            description="Get weather information for a location. Use location parameter."
        )
        # TODO: Add mock weather data
        self.weather_data = {}
    
    def execute(self, location: str = "", **kwargs) -> str:
        """Execute weather lookup."""
        # TODO: Implement weather lookup
        return f"Weather data for {location}"

# TODO: Test your new tool
print("Exercise area - add your implementation here!")